# Local inner-loop adapter: guided implementation

This notebook is the implementation workbench for the first real Carrot Tuning component. It works against the actual adapter source and the source-controlled historical contract. It does not replace the runner with notebook-only logic.

We will work one reviewed step at a time: make a small change to the Python adapter, run the relevant cell here, inspect a separate human-readable result, and then decide what belongs in the next step.

## Boundaries for this workbench

- The adapter is local only and calls the proven parent evaluator only after validation.
- The historical record is a reproduction oracle, not automatic permission to run it.
- Candidate values, source-policy changes, Foundry, Azure, and external services remain outside this notebook unless separately approved.
- A notebook cell may verify the adapter's behavior, but the reusable behavior belongs in the Python files under evaluator/.

## 9.1 Read the completed versioned experiment specification

The contract and schema already exist. This setup cell finds them from the workspace, loads the historical JSON for inspection, and makes the real adapter package importable. It does not validate the schema or execute a simulator.

In [ ]:
import json
import sys
from pathlib import Path


def find_workspace_file(relative_path: Path) -> Path:
    """Find a checked-in project file when the notebook server chooses a different cwd."""
    for directory in (Path.cwd(), *Path.cwd().parents):
        candidate = directory / relative_path
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Could not find workspace file: {relative_path}")


# The adapter owns reusable behavior, so this notebook imports it rather than copying it.
adapter_source = find_workspace_file(Path("carrot-tuning-system/evaluator/runner.py"))
project_root = adapter_source.parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# This is the completed historical contract that the future runner must validate and reproduce.
spec_path = find_workspace_file(
    Path("carrot-tuning-system/experiments/historical/carrot_investment/seed_opportunity_coarse_v1.json")
)
schema_path = find_workspace_file(
    Path("carrot-tuning-system/experiments/schema/experiment-spec-v1.schema.json")
)
historical_spec = json.loads(spec_path.read_text())

# The next cell presents loaded facts; it performs no schema validation or evaluator call.

In [1]:
from IPython.display import Markdown, display


def render_contract_checkpoint(spec: dict, source: Path) -> None:
    """Presentation only: clearing this output never changes the loaded contract."""
    parameter = spec["parameter"]
    candidates = ", ".join(str(value) for value in parameter["candidate_values"])
    display(Markdown(
        f"""### Contract loaded for implementation work

| Item | Checked-in source |
|:---|:---|
| Historical specification | {spec['experiment_id']} |
| Schema version | {spec['schema_version']} |
| Lifecycle | {spec['lifecycle_status']} |
| Parameter envelope | {parameter['name']}: baseline {parameter['baseline_value']:g}; candidates {candidates} |
| Adapter source | {source.parent.name}/{source.name} |

> Loaded is not validated. The first adapter implementation step is to validate this contract before any target binding or simulator call.
"""
    ))


render_contract_checkpoint(historical_spec, adapter_source)

### Contract loaded for implementation work

| Item | Checked-in source |
|:---|:---|
| Historical specification | carrot-seed-opportunity-coarse-v1 |
| Schema version | 1 |
| Lifecycle | historical_reproduction |
| Parameter envelope | seed_opportunity_value: baseline 20; candidates 10, 40, 60, 80 |
| Adapter source | evaluator/runner.py |

> Loaded is not validated. The first adapter implementation step is to validate this contract before any target binding or simulator call.


## 9.2 Implement a local inner-loop adapter

The real code lives in evaluator/runner.py. Before we add its first implementation step, this cell proves that the current framework is safely inert. It exercises the actual Python entry point and expects the deliberate stop, not a simulator run.

In [ ]:
from evaluator import LocalInnerLoopAdapter, LocalRunRequest


# This request describes future locations only. The framework must not use them yet.
framework_request = LocalRunRequest(
    spec_path=spec_path,
    target_root=project_root.parent / "kaggriculture",
    report_directory=project_root.parent / "var" / "carrot-tuning-system",
)

# Calling the real entry point should fail closed until the first reviewed stage is implemented.
try:
    LocalInnerLoopAdapter().run(framework_request)
except NotImplementedError as error:
    framework_state = str(error)
else:
    raise AssertionError("The adapter unexpectedly ran before implementation was reviewed.")

# The next cell presents this checkpoint without exposing the implementation details.

In [2]:
from IPython.display import Markdown, display


def render_adapter_checkpoint(message: str) -> None:
    """Presentation only: clearing this output never changes adapter behavior."""
    display(Markdown(
        f"""### Adapter checkpoint

| Check | Current result |
|:---|:---|
| Imported reusable adapter | yes |
| Called real run entry point | yes |
| Simulator, target binding, or report write | no |
| Current state | {message} |

> This is the intended starting control: the adapter must stay fail-closed until we implement and review its next smallest step.
"""
    ))


render_adapter_checkpoint(framework_state)

### Adapter checkpoint

| Check | Current result |
|:---|:---|
| Imported reusable adapter | yes |
| Called real run entry point | yes |
| Simulator, target binding, or report write | no |
| Current state | scaffolded only; schema validation is next |

> This is the intended starting control: the adapter must stay fail-closed until we implement and review its next smallest step.


## 9.3 Persist the evidence packet

Not started. After the adapter can validate an approved contract and produce audited match records, we will define a durable, local ignored evidence-packet shape here. It must include provenance, per-match results, paired aggregates, divergence and suffix facts, timing components, and audit status. We will not invent a report before there is real runner output to preserve.

## 9.4 Verify reproduction before adding intelligence

Not started. The first real run must reproduce the historical classifications: 10 inert at scale; 40, 60, and 80 directional harm; all required clean audits passing. A mismatch is a runner repair task, not an invitation to alter the candidate envelope or explain the result away.

## 9.5 Preserve the broader roadmap

This notebook stays focused on the local adapter. Only after the reproduction smoke test succeeds do we move to a graph-adapter contract, then deterministic outer evaluation, and finally a Foundry-facing reasoning adapter. None of those later components score the game or edit policy source.

## Next reviewed change

Implement only the schema-and-lifecycle validation stage in evaluator/runner.py, then return here to exercise that real code against the historical JSON. Do not bind Kaggriculture or run a game in that change.